# ASTROF — Training Notebook
**Autonomous Scheduling Through Role-Oriented Federation**

This notebook trains the ASTROF multi-agent telescope scheduling system using:
- **SFT warm-start** on 5,780 greedy-policy demonstrations
- **GRPO curriculum** (easy → medium → hard) via TRL GRPOTrainer
- **Continual learning loop** with NoveltyDetector + LoRA adapter merging

**Requirements:** A100 40GB GPU (Colab Pro / on-site hardware)

**Model:** `unsloth/Qwen3-1.7B-unsloth-bnb-4bit` (4-bit quantized, fits in 40GB with room for GRPO rollouts)

**Environment:** Live at `https://Cosmosatom-astrof.hf.space`

## 0. GPU Check

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

## 1. Install Dependencies

In [ ]:
%%capture
# Core training stack
!pip install unsloth trl peft datasets transformers accelerate bitsandbytes

# OpenEnv client
!pip install openenv-client

# Astronomy + API
!pip install astropy openai requests

## 2. Clone Repository

In [ ]:
import os

REPO_URL = "https://github.com/Cosmosatom/astrof"  # update if repo URL differs
REPO_DIR = "astrof"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
!ls

## 3. Configuration

Set your HF token to push the trained adapter to the Hub after training.

In [ ]:
import os

# --- Required ---
os.environ["ENV_BASE_URL"]  = "https://Cosmosatom-astrof.hf.space"  # live environment
os.environ["MODEL_NAME"]    = "unsloth/Qwen3-1.7B-unsloth-bnb-4bit"

# --- Optional: set to push adapter to HF Hub after training ---
# os.environ["HF_TOKEN"]   = "hf_xxx"           # your HF write token
# os.environ["HF_REPO_ID"] = "Cosmosatom/astrof-adapter"  # target repo

print("Config set:")
print(f"  ENV_BASE_URL : {os.environ['ENV_BASE_URL']}")
print(f"  MODEL_NAME   : {os.environ['MODEL_NAME']}")

## 4. Smoke Test

Verify imports, data files, and environment connectivity before starting training.

In [ ]:
!python scripts/smoke_test.py

## 5. Load Model (Qwen3-1.7B, 4-bit)

Loads the base model with Unsloth optimizations and attaches a LoRA adapter (r=16).

In [ ]:
import sys, os
sys.path.insert(0, "scripts")

from unsloth import FastLanguageModel

MODEL_NAME = os.environ["MODEL_NAME"]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    dtype=None,          # auto-detect bf16 on A100
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=2026,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 6. SFT Warm-start

Fine-tunes one epoch on 5,780 greedy-policy demonstrations (`data/sft_warmstart.jsonl`).
This primes the model to output valid JSON for all 5 role-conditioned prompts before RL begins.
Without this, GRPO sees only zero-reward rollouts in early training.

**Expected time:** ~10 min on A100

In [ ]:
from train_grpo import sft_warmstart

sft_warmstart(model, tokenizer)
print("SFT warm-start complete.")

## 7. GRPO Curriculum Training

Trains the model using Group Relative Policy Optimization across 3 difficulty levels:

| Stage | Task | What the agent learns |
|-------|------|-----------------------|
| 1 | easy | Single telescope, JSON format, priority scoring |
| 2 | medium | 3-telescope coordination, duplicate avoidance |
| 3 | hard | Weather recovery, transit deadlines |

**Reward:** `0.9 × science_yield + 0.1 × parse_rate`  
**Expected time:** ~20 min per stage on A100 (100 steps each)

In [ ]:
from train_grpo import train_grpo, CURRICULUM

print(f"Curriculum: {CURRICULUM}")

for task_id in CURRICULUM:
    print(f"\n{'='*50}")
    print(f"Training on task: {task_id}")
    print(f"{'='*50}")
    train_grpo(model, tokenizer, task_id)

## 8. (Optional) Expert Task — ToO + Novel Category

Run this if time allows. Tests Target-of-Opportunity interrupt handling and
in-context adaptation to the `gravitational_wave_host` category.

**Expected time:** ~20 min on A100

In [ ]:
from train_grpo import train_grpo

train_grpo(model, tokenizer, "expert")
print("Expert task training complete.")

## 9. Continual Learning Loop

Runs 200 post-curriculum episodes on hard+expert tasks.
The `NoveltyDetector` watches every step — when 30 novel episodes accumulate,
`ContinualLearner` automatically trains a new LoRA adapter and merges it into the model.

No human intervention required.

**Expected time:** ~30 min on A100 (may trigger 1-2 adapter merges)

In [ ]:
from train_grpo import run_continual_loop

model = run_continual_loop(model, tokenizer)
print("Continual learning loop complete.")

## 10. Save Adapter

In [ ]:
import os

os.makedirs("outputs/final", exist_ok=True)
model.save_pretrained("outputs/final")
tokenizer.save_pretrained("outputs/final")
print("Adapter saved to outputs/final/")
!ls outputs/final/

## 11. Push to HF Hub (Optional)

In [ ]:
HF_TOKEN   = os.environ.get("HF_TOKEN", "")
HF_REPO_ID = os.environ.get("HF_REPO_ID", "")

if HF_TOKEN and HF_REPO_ID:
    model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    print(f"Pushed to {HF_REPO_ID}")
else:
    print("HF_TOKEN or HF_REPO_ID not set — skipping push.")
    print("Set os.environ['HF_TOKEN'] and os.environ['HF_REPO_ID'] to enable.")

## 12. Benchmark — Before vs After

Run the benchmark script with the trained model to get your AFTER numbers.
Compare against the zero-shot baseline already collected.

**Requires:** Ollama running with the trained model, OR set `API_BASE_URL` to HF Inference.

In [ ]:
# Option A: run inference directly in this notebook using the trained model
import sys, os, json, re
sys.path.insert(0, ".")
sys.path.insert(0, "scripts")

from train_grpo import _make_direct_model_fn, _run_episode, MAX_STEPS_PER_TASK

model_fn = _make_direct_model_fn(model, tokenizer)

print("Running trained model evaluation...")
results = {}
for task in ["easy", "medium", "hard", "expert"]:
    steps = _run_episode(task, model_fn)
    rewards = [s["reward"] for s in steps]
    avg_reward = sum(rewards) / max(len(rewards), 1)
    results[task] = round(avg_reward, 4)
    print(f"  {task:8s}  avg_reward={avg_reward:.4f}  steps={len(rewards)}")

print("\nTrained model results:")
print(json.dumps(results, indent=2))

## Summary

| Stage | What ran |
|-------|----------|
| SFT warm-start | 5,780 greedy demonstrations, 1 epoch |
| GRPO easy | Single telescope, format + priority learning |
| GRPO medium | 3-telescope coordination, duplicate avoidance |
| GRPO hard | Weather recovery, transit deadlines |
| GRPO expert | ToO interrupts, novel category adaptation |
| Continual loop | 200 episodes, auto LoRA on novelty |

**Adapter saved to:** `outputs/final/`

---
*ASTROF — Team X: Kavya Sree Kammari, Sanyam Bhardwaj, Yasasree Lasya*  
*Meta PyTorch / OpenEnv Hackathon — Grand Finale, April 2026*